# Chapter 2: Pre-Model Workflow and Data Preprocessing

**Buku:** scikit-learn Cookbook (O'Reilly)
**Topik:** Alur kerja sebelum modeling dan preprocessing data

## Ringkasan Chapter

Kualitas data sangat menentukan kualitas model ("*garbage in, garbage out*"). Sebelum data masuk ke model, biasanya ada beberapa masalah yang harus ditangani:

| Masalah | Alat di scikit-learn |
|:--|:--|
| Data hilang (*missing values*) | `SimpleImputer`, `KNNImputer`, `IterativeImputer` |
| Skala fitur berbeda-beda | `StandardScaler`, `MinMaxScaler`, `Normalizer`, `RobustScaler` |
| Fitur kategorikal | `OneHotEncoder`, `OrdinalEncoder`, `LabelEncoder`, `ColumnTransformer` |
| Alur kerja yang rawan *data leakage* | `Pipeline` |
| Fitur kurang informatif | `PolynomialFeatures`, `KBinsDiscretizer`, `RFE`, `SelectFromModel` |

> Cara pakai: jalankan sel dari atas ke bawah. Setiap sel kode diikuti penjelasan teori dan cara membaca hasilnya.

In [ ]:
import numpy as np
import pandas as pd
import sklearn
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)

print("scikit-learn:", sklearn.__version__)
print("NumPy       :", np.__version__)
print("pandas      :", pd.__version__)

**Penjelasan:** sel ini memuat library dan mengatur tampilan pandas (maksimal 15 kolom, 4 angka desimal) supaya tabel mudah dibaca. Mengecek versi berguna karena perilaku beberapa fungsi bisa berbeda antar versi.

## 1. Mengapa Preprocessing Penting?

Masalah data yang paling sering muncul:

1. **Data hilang:** sebagian besar estimator scikit-learn menolak input yang berisi `NaN`.
2. **Outlier:** nilai ekstrem bisa menarik parameter model jauh dari pola sebenarnya.
3. **Variabel kategorikal:** model hanya memahami angka, jadi teks harus dikodekan.
4. **Skala fitur berbeda:** fitur dengan angka besar mendominasi fitur berangka kecil pada algoritma berbasis jarak atau gradien.
5. **Data leakage:** informasi dari data uji ikut dipakai saat preprocessing sehingga skor evaluasi terlihat lebih bagus daripada kenyataan.

Untuk melihat dampak nyata poin 4, kita bandingkan KNN pada dataset *wine* dengan dan tanpa penskalaan.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

Xw, yw = load_wine(return_X_y=True, as_frame=True)
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(
    Xw, yw, test_size=0.3, random_state=42, stratify=yw
)

# Kisaran nilai tiap fitur (sangat berbeda-beda)
print(Xw.describe().loc[["min", "max"]].T.head(6))

knn_raw = KNeighborsClassifier().fit(Xw_tr, yw_tr)
knn_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier()).fit(Xw_tr, yw_tr)

print(f"\nAkurasi KNN tanpa scaling : {knn_raw.score(Xw_te, yw_te):.4f}")
print(f"Akurasi KNN dengan scaling: {knn_scaled.score(Xw_te, yw_te):.4f}")

**Penjelasan hasil:** pada dataset ini fitur `proline` bernilai ratusan sampai ribuan, sedangkan fitur lain hanya berkisar puluhan atau di bawah 10. KNN menghitung jarak, sehingga tanpa scaling jarak hampir sepenuhnya ditentukan oleh `proline`. Setelah diskalakan, semua fitur berkontribusi seimbang dan akurasinya naik jelas. Ini alasan utama scaling dilakukan pada algoritma berbasis jarak.

## 2. Menangani Data Hilang (*Missing Data*)

Tiga metode imputasi di scikit-learn:

| Metode | Cara kerja | Cocok untuk |
|:--|:--|:--|
| `SimpleImputer` | Isi dengan mean/median/modus/konstanta | Cepat, sederhana, data hilang sedikit |
| `KNNImputer` | Isi dengan rata-rata tetangga terdekat | Data numerik dengan struktur lokal |
| `IterativeImputer` | Memodelkan tiap fitur dengan regresi dari fitur lain | Hubungan antar fitur yang kuat |

### 2.1 Membuat dataset dengan nilai hilang

Kita buat 20 sampel dan 10 fitur, lalu sekitar 20% nilainya dikosongkan secara acak.

In [ ]:
np.random.seed(2024)
n_samples, n_features = 20, 10

df_full = pd.DataFrame({
    f"Feature{i+1}": np.random.uniform(0, 100, n_samples) for i in range(n_features)
})

df = df_full.copy()
for col in df.columns:
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, col] = np.nan

total = df.size
missing = int(df.isna().sum().sum())
print(f"Ukuran data   : {df.shape[0]} sampel x {df.shape[1]} fitur")
print(f"Total sel     : {total}")
print(f"Sel hilang    : {missing} ({missing/total*100:.1f}%)")
print("\nHilang per fitur:")
print(df.isna().sum().to_string())
print("\n6 baris pertama:")
print(df.head(6).to_string())

**Penjelasan hasil:** persentase data hilang sekitar 20% (nilainya bisa sedikit di atas atau di bawah 20 karena pengosongan dilakukan acak). Pada kondisi nyata, pola data hilang jarang merata antar fitur. Jika DataFrame ini langsung diberikan ke model seperti `LinearRegression`, akan muncul `ValueError`, jadi imputasi wajib dilakukan lebih dulu.

### 2.2 SimpleImputer

`SimpleImputer` mengisi setiap `NaN` dengan statistik kolom yang dihitung dari nilai yang ada:

- `"mean"` (default): rata-rata kolom $\mu_j = \dfrac{1}{n_{obs}}\sum x_{ij}$
- `"median"`: median, lebih tahan terhadap outlier
- `"most_frequent"`: modus, cocok untuk kategorikal
- `"constant"`: nilai tetap pilihan kita (`fill_value`)

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

print("Mean yang dipelajari tiap fitur:")
print(pd.Series(imputer.statistics_, index=df.columns).round(2).to_string())
print("\nNaN tersisa:", int(imputed.isna().sum().sum()))
print("\n6 baris pertama setelah imputasi:")
print(imputed.head(6).to_string())

**Penjelasan hasil:** semua `NaN` pada satu kolom diganti dengan angka yang sama, yaitu mean kolom itu (tersimpan di `imputer.statistics_`). Kelebihannya cepat dan mudah dipahami. Kekurangannya, **variansi fitur mengecil** karena banyak nilai ditarik ke pusat, sehingga korelasi antar fitur bisa melemah.

Mari bandingkan strategi lain dalam satu contoh kecil:

In [ ]:
mini = pd.DataFrame({"umur": [20, 25, np.nan, 40, 100],
                     "kota": ["A", "B", "A", np.nan, "A"]})

print("Data kecil:\n", mini, "\n")

num = SimpleImputer(strategy="mean").fit_transform(mini[["umur"]]).ravel()
med = SimpleImputer(strategy="median").fit_transform(mini[["umur"]]).ravel()
print("umur (mean)  :", num)
print("umur (median):", med)

cat = SimpleImputer(strategy="most_frequent").fit_transform(mini[["kota"]]).ravel()
print("kota (modus) :", cat)

const = SimpleImputer(strategy="constant", fill_value="Tidak diketahui").fit_transform(mini[["kota"]]).ravel()
print("kota (konstan):", const)

**Penjelasan hasil:** karena ada outlier (umur 100), mean terisi jauh lebih besar daripada median. Inilah alasan median dianjurkan bila data punya nilai ekstrem. Untuk kolom teks, `mean` tidak bisa dipakai sehingga digunakan `most_frequent` atau `constant`.

### 2.3 KNNImputer

`KNNImputer` mencari $k$ sampel terdekat (jarak Euclidean pada fitur yang sama-sama terisi) lalu mengisi nilai hilang dengan rata-rata tetangganya:

$$d(\mathbf{x}_i, \mathbf{x}_j) = \sqrt{\frac{p}{p_{obs}} \sum_{l \in \text{observed}} (x_{il} - x_{jl})^2}$$

dengan $p$ jumlah seluruh fitur dan $p_{obs}$ jumlah fitur yang terisi pada kedua sampel (pengali $p/p_{obs}$ mengimbangi fitur yang tidak ikut dihitung).

Hasilnya bersifat **lokal**: tiap sampel bisa mendapat nilai pengisi yang berbeda.

In [ ]:
from sklearn.impute import KNNImputer

knn_imp = KNNImputer(n_neighbors=2)
knn_imputed = pd.DataFrame(knn_imp.fit_transform(df), columns=df.columns)

print("NaN tersisa:", int(knn_imputed.isna().sum().sum()))

idx = df[df["Feature1"].isna()].index.tolist()
print(f"\nFeature1 hilang pada baris: {idx}")
print("  SimpleImputer :", imputed.loc[idx, "Feature1"].round(2).tolist())
print("  KNNImputer    :", knn_imputed.loc[idx, "Feature1"].round(2).tolist())

**Penjelasan hasil:** `SimpleImputer` mengisi semua baris dengan angka sama, sedangkan `KNNImputer` memberi angka berbeda-beda sesuai tetangga masing-masing baris. Biayanya lebih mahal karena harus menghitung jarak antar sampel (kira-kira $O(n^2 p)$), dan hasilnya sensitif terhadap pilihan `n_neighbors`: terlalu kecil menghasilkan varians tinggi, terlalu besar menghaluskan pola lokal. Karena berbasis jarak, **fitur sebaiknya diskalakan lebih dulu** bila skalanya berbeda.

### 2.4 IterativeImputer

`IterativeImputer` memodelkan tiap fitur yang punya `NaN` sebagai fungsi regresi dari fitur lain, lalu mengulanginya:

1. Isi awal semua `NaN` (misalnya dengan mean).
2. Untuk tiap fitur $j$: jadikan fitur $j$ sebagai target, fitur lain sebagai prediktor, latih regresi, dan prediksi nilai yang hilang.
3. Ulangi sampai konvergen atau mencapai `max_iter`.

Idenya mirip algoritma **MICE** dari statistika. Fitur ini masih berstatus *experimental*, jadi harus diaktifkan secara eksplisit.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # wajib diimpor dulu
from sklearn.impute import IterativeImputer

it_imp = IterativeImputer(random_state=2024, max_iter=10)
it_imputed = pd.DataFrame(it_imp.fit_transform(df), columns=df.columns)

print("NaN tersisa:", int(it_imputed.isna().sum().sum()))
print(f"\nPerbandingan nilai pengisi Feature1 (baris {idx}):")
print("  SimpleImputer    :", imputed.loc[idx, "Feature1"].round(2).tolist())
print("  KNNImputer (k=2) :", knn_imputed.loc[idx, "Feature1"].round(2).tolist())
print("  IterativeImputer :", it_imputed.loc[idx, "Feature1"].round(2).tolist())

**Penjelasan hasil:** tiga metode memberi tiga nilai berbeda untuk sel yang sama. Artinya **imputasi adalah estimasi, bukan nilai asli**. Tiap metode punya asumsi sendiri.

Karena data kita dibuat acak dan antar fitur tidak berhubungan, jangan berharap `IterativeImputer` jauh lebih baik di sini. Metode ini unggul bila antar fitur memang ada hubungan. Kita buktikan di sel berikutnya.

### 2.5 Mana yang lebih akurat? (eksperimen tambahan)

Karena kita punya data asli sebelum dikosongkan (`df_full`), kita bisa mengukur error tiap metode pada sel yang dikosongkan. Kita uji pada dua data: acak (tanpa hubungan antar fitur) dan berkorelasi.

In [ ]:
def rmse_imputers(clean, frac=0.2, seed=0):
    rng = np.random.RandomState(seed)
    X = clean.values.copy()
    mask = rng.random(X.shape) < frac
    X_miss = X.copy()
    X_miss[mask] = np.nan
    out = {}
    for name, imp in {
        "Simple (mean)": SimpleImputer(strategy="mean"),
        "KNN (k=3)": KNNImputer(n_neighbors=3),
        "Iterative": IterativeImputer(random_state=0, max_iter=10),
    }.items():
        filled = imp.fit_transform(X_miss)
        out[name] = np.sqrt(np.mean((filled[mask] - X[mask]) ** 2))
    return out

# Data A: fitur saling independen
rng = np.random.RandomState(1)
A = pd.DataFrame(rng.normal(size=(300, 5)), columns=list("abcde"))

# Data B: fitur saling berkorelasi kuat
base = rng.normal(size=(300, 1))
B = pd.DataFrame(base + 0.3 * rng.normal(size=(300, 5)), columns=list("abcde"))

res = pd.DataFrame({"Data independen": rmse_imputers(A), "Data berkorelasi": rmse_imputers(B)})
print("RMSE imputasi (makin kecil makin baik):")
print(res.round(4).to_string())

**Penjelasan hasil:** pada data independen ketiga metode berhasil kurang lebih sama, dan KNN bahkan sedikit lebih buruk daripada mean, karena tidak ada informasi dari fitur lain yang bisa dimanfaatkan (tetangga terdekat pun tidak lebih informatif). Pada data berkorelasi, `IterativeImputer` (dan biasanya KNN) memberi error jauh lebih kecil dibanding mean. Jadi pemilihan metode imputasi sebaiknya mempertimbangkan apakah antar fitur ada hubungan.

**Panduan praktis:**
- Data hilang < 5%: `SimpleImputer` biasanya cukup.
- Data hilang 5-10% dan fitur numerik: coba `KNNImputer` atau `IterativeImputer`.
- Satu fitur hilang > 10%: pertimbangkan membuang fitur tersebut atau gunakan pengetahuan domain.
- Beberapa model berbasis pohon (misalnya `HistGradientBoosting`) bisa menangani `NaN` secara langsung.

## 3. Teknik Penskalaan (*Scaling*)

Perbedaan skala menimbulkan dua masalah:

1. **Algoritma berbasis jarak** (KNN, SVM, K-Means) didominasi fitur berangka besar.
2. **Algoritma berbasis gradien** (regresi logistik, neural network) lebih lambat konvergen bila skala fitur timpang.

| Scaler | Rumus | Rentang hasil | Cocok untuk |
|:--|:--|:--|:--|
| `StandardScaler` | $z = \dfrac{x-\mu}{\sigma}$ | Berpusat di 0 | Fitur mendekati normal |
| `MinMaxScaler` | $x' = \dfrac{x-x_{min}}{x_{max}-x_{min}}$ | $[0,1]$ | Fitur terbatas, neural network |
| `RobustScaler` | $x' = \dfrac{x-\text{median}}{\text{IQR}}$ | Berpusat di 0 | Data dengan outlier |
| `Normalizer` | $x' = \dfrac{x}{\lVert x \rVert}$ | Norma 1 per baris | Data sparse, teks |

**Perbedaan penting:** `StandardScaler`, `MinMaxScaler`, dan `RobustScaler` bekerja **per kolom (fitur)**, sedangkan `Normalizer` bekerja **per baris (sampel)**.

### 3.1 StandardScaler

In [ ]:
scaler = StandardScaler()
std_df = pd.DataFrame(scaler.fit_transform(it_imputed), columns=it_imputed.columns)

print("Mean dipelajari :", np.round(scaler.mean_, 2))
print("Std dipelajari  :", np.round(scaler.scale_, 2))
print("\nSetelah scaling:")
print("  Mean tiap kolom:", np.round(std_df.mean().values, 8))
print("  Std tiap kolom :", np.round(std_df.std(ddof=0).values, 4))

# Verifikasi manual untuk sampel pertama Feature1
x0 = it_imputed.loc[0, "Feature1"]
manual = (x0 - scaler.mean_[0]) / scaler.scale_[0]
print(f"\nz-score manual Feature1 baris 0: {manual:.4f} | hasil scaler: {std_df.loc[0, 'Feature1']:.4f}")

**Penjelasan hasil:** setiap kolom kini bermean 0 dan standar deviasi 1 (angka seperti `1e-17` hanya sisa pembulatan floating point). Hitungan manual rumus $z=(x-\mu)/\sigma$ sama persis dengan hasil `StandardScaler`. Nilai z-score bisa negatif dan bisa melebihi $\pm1$, karena standarisasi tidak membatasi rentang. Parameter $\mu$ dan $\sigma$ tersimpan di `mean_` dan `scale_`, dan itulah yang dipakai ulang saat `transform()` pada data uji.

### 3.2 MinMaxScaler

In [ ]:
from sklearn.preprocessing import MinMaxScaler

mm = MinMaxScaler()                      # default feature_range=(0, 1)
mm_df = pd.DataFrame(mm.fit_transform(it_imputed), columns=it_imputed.columns)

print("Min asli tiap fitur:", np.round(mm.data_min_, 2))
print("Max asli tiap fitur:", np.round(mm.data_max_, 2))
print("\nSetelah scaling:")
print("  Min:", mm_df.min().values.round(4))
print("  Max:", mm_df.max().values.round(4))

# Rentang kustom
mm2 = MinMaxScaler(feature_range=(-1, 1))
print("\nRentang kustom (-1, 1) -> min/max Feature1:",
      mm2.fit_transform(it_imputed[["Feature1"]]).min().round(2),
      mm2.fit_transform(it_imputed[["Feature1"]]).max().round(2))

**Penjelasan hasil:** nilai terkecil tiap kolom menjadi tepat 0 dan terbesar tepat 1, sisanya diinterpolasi linear. Rentang bisa diubah lewat `feature_range`.

Kelemahan utama `MinMaxScaler` adalah **sensitif terhadap outlier**: satu nilai ekstrem meregangkan rentang sehingga data lainnya terhimpit di area sempit. Demonstrasinya ada pada bagian 3.4.

### 3.3 Normalizer (per baris)

In [ ]:
from sklearn.preprocessing import Normalizer

norm = Normalizer(norm="l2")
norm_df = pd.DataFrame(norm.fit_transform(it_imputed), columns=it_imputed.columns)

row_norms = np.sqrt((norm_df ** 2).sum(axis=1))
print("Norma L2 tiap baris (harus 1):")
print(row_norms.round(6).values)

norm_l1 = Normalizer(norm="l1").fit_transform(it_imputed)
print("\nJumlah |nilai| tiap baris pada norma L1 (5 baris pertama):", np.abs(norm_l1).sum(axis=1)[:5].round(6))

**Penjelasan hasil:** `Normalizer` membagi setiap **baris** dengan normanya, sehingga setiap sampel memiliki panjang 1 (berada di permukaan hipersfera satuan). Pada norma L1, jumlah nilai absolut tiap baris menjadi 1.

`Normalizer` tidak mempelajari apa pun dari data (`fit()` tidak melakukan apa-apa), karena hitungannya hanya bergantung pada baris itu sendiri. Penggunaannya umum pada data teks (TF-IDF) agar panjang dokumen tidak memengaruhi perbandingan kemiripan.

### 3.4 Perbandingan saat ada outlier: Standard vs MinMax vs Robust

In [ ]:
from sklearn.preprocessing import RobustScaler

gaji = np.array([[50], [55], [60], [62], [58], [65], [53], [10000]], dtype=float)  # satu outlier

hasil = pd.DataFrame({
    "Asli": gaji.ravel(),
    "Standard": StandardScaler().fit_transform(gaji).ravel(),
    "MinMax": MinMaxScaler().fit_transform(gaji).ravel(),
    "Robust": RobustScaler().fit_transform(gaji).ravel(),
})
print(hasil.round(4).to_string(index=False))

**Penjelasan hasil:** pada `MinMaxScaler`, tujuh nilai normal terhimpit mendekati 0 dan hanya outlier yang bernilai 1. `StandardScaler` juga terpengaruh karena mean dan standar deviasi ikut tertarik outlier. `RobustScaler` memakai median dan IQR, sehingga tujuh nilai normal tetap tersebar wajar, sementara outlier muncul sebagai angka sangat besar yang terlihat jelas. Jadi bila data punya outlier, `RobustScaler` biasanya pilihan yang lebih aman.

## 4. Encoding Variabel Kategorikal

Model membutuhkan angka, sehingga kategori harus dikodekan. Pilihan encoder bergantung pada jenis variabel:

- **Nominal** (tanpa urutan, misalnya kota atau departemen): `OneHotEncoder`.
- **Ordinal** (ada urutan, misalnya Junior < Senior < Manager): `OrdinalEncoder` dengan urutan yang kita tentukan.
- **Target/label** (kolom `y`): `LabelEncoder`.

Salah memilih encoder menciptakan **hubungan urutan palsu**. Misalnya Red=0, Green=1, Blue=2 akan dibaca model linear seolah Blue "dua kali" Green.

### 4.1 Membuat dataset kategorikal

In [ ]:
np.random.seed(2024)
cat_df = pd.DataFrame({
    "Department": np.random.choice(["A", "B", "C", "D"], size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
    "Location": np.random.choice(["NY", "SF", "LA", "CHI"], size=20),
})

print("Ukuran:", cat_df.shape)
for col in cat_df.columns:
    print(f"  {col}: {sorted(cat_df[col].unique())}")
print()
print(cat_df.head(8).to_string())

**Penjelasan:** ada tiga fitur kategorikal dengan masing-masing 4, 3, dan 4 kategori. `Position` sebenarnya ordinal (ada urutan jabatan), sedangkan `Department` dan `Location` nominal.

### 4.2 OneHotEncoder

One-hot membuat satu kolom biner per kategori. Untuk $K$ kategori, dihasilkan $K$ kolom (atau $K-1$ jika `drop="first"`):

$$\text{OHE}(x_i = c) = \begin{cases}1 & x_i = c \\ 0 & \text{lainnya}\end{cases}$$

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False)
ohe_df = pd.DataFrame(ohe.fit_transform(cat_df), columns=ohe.get_feature_names_out())

print("Ukuran asli  :", cat_df.shape)
print("Ukuran encode:", ohe_df.shape)
print("Kolom baru   :", list(ohe_df.columns))
print()
print(ohe_df.head(5).astype(int).to_string())

print("\nJumlah angka 1 per baris (harus = jumlah fitur asli, yaitu 3):",
      set(ohe_df.sum(axis=1).astype(int)))

# Opsi: drop='first' untuk menghindari multikolinearitas
ohe_drop = OneHotEncoder(sparse_output=False, drop="first")
print("\nUkuran dengan drop='first':", ohe_drop.fit_transform(cat_df).shape)

**Penjelasan hasil:** tiga kolom asli menjadi 4 + 3 + 4 = **11** kolom biner. Setiap baris memiliki tepat satu angka 1 untuk tiap fitur asli. Dengan `drop="first"` satu kolom per fitur dibuang (menjadi 8 kolom) karena kolom terakhir bisa disimpulkan dari yang lain. Hal ini berguna pada model linear untuk menghindari multikolinearitas.

**Masalah kardinalitas tinggi:** fitur dengan ribuan kategori menghasilkan ribuan kolom. Alternatifnya adalah *target encoding*, *frequency encoding*, atau *hashing*.

**Menangani kategori baru saat prediksi:** secara default `OneHotEncoder` akan error bila data baru berisi kategori yang tidak pernah dilihat saat `fit`. Gunakan `handle_unknown="ignore"` agar kategori asing diubah menjadi semua nol.

In [ ]:
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
enc.fit(pd.DataFrame({"kota": ["NY", "SF", "LA"]}))

baru = pd.DataFrame({"kota": ["NY", "Tokyo"]})   # Tokyo belum pernah dilihat
print("Kategori dikenal:", enc.categories_[0])
print(enc.transform(baru))

**Penjelasan hasil:** "Tokyo" tidak dikenal sehingga seluruh barisnya bernilai 0, dan program tidak error. Pola ini penting pada sistem produksi.

### 4.3 OrdinalEncoder (untuk variabel yang punya urutan)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(categories=[["Junior", "Senior", "Manager"]])   # urutan ditentukan manual
pos_enc = oe.fit_transform(cat_df[["Position"]])

print("Urutan kategori:", oe.categories_[0])
print(pd.DataFrame({"Position": cat_df["Position"], "Encoded": pos_enc.ravel().astype(int)}).head(8).to_string(index=False))

**Penjelasan hasil:** Junior=0, Senior=1, Manager=2 sesuai urutan yang kita tentukan. Bila `categories` tidak diisi, scikit-learn mengurutkan secara alfabet (Junior, Manager, Senior), yang tidak sesuai urutan jabatan. Maka untuk data ordinal, **selalu tentukan urutan secara eksplisit**.

### 4.4 LabelEncoder (khusus untuk target)

`LabelEncoder` memberi bilangan bulat 0, 1, 2, ... pada tiap kelas secara alfabetis. Dokumentasi scikit-learn menyatakan encoder ini dirancang untuk **kolom target `y`**, bukan fitur `X`.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_text = np.array(["lulus", "gagal", "lulus", "lulus", "gagal"])
y_num = le.fit_transform(y_text)

print("Teks     :", y_text)
print("Angka    :", y_num)
print("Kelas    :", le.classes_)
print("Balik    :", le.inverse_transform(y_num))

# Efek buruk bila dipakai pada fitur nominal
for col in cat_df.columns:
    m = {c: i for i, c in enumerate(sorted(cat_df[col].unique()))}
    print(f"Pemetaan {col}: {m}")

**Penjelasan hasil:** `classes_` menyimpan urutan kelas, dan `inverse_transform()` mengembalikan angka menjadi label asli, berguna untuk menampilkan hasil prediksi. Pemetaan di bagian bawah menunjukkan bahaya memakai bilangan bulat pada fitur nominal: `Department` D=3 terlihat "tiga kali" A=0, padahal tidak bermakna.

**Kapan label integer aman?** Pada model berbasis pohon, karena pohon hanya memakai ambang batas (apakah nilai $\leq 1.5$?) dan bukan jarak. Pada model linear, SVM, atau neural network, gunakan one-hot untuk fitur nominal.

### 4.5 ColumnTransformer

Dataset nyata biasanya campuran kolom numerik dan kategorikal yang butuh perlakuan berbeda. `ColumnTransformer` menerapkan transformer berbeda ke kolom berbeda dalam satu objek.

In [ ]:
from sklearn.compose import ColumnTransformer

np.random.seed(2024)
mixed = pd.DataFrame({
    "Age": np.random.randint(25, 65, size=20),
    "Salary": np.round(np.random.normal(60000, 15000, size=20), 2),
    "Experience": np.random.randint(1, 20, size=20),
    "Department": np.random.choice(["IT", "HR", "Sales", "Finance"], size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
})
print(mixed.head(6).to_string())
print("\nTipe data:\n", mixed.dtypes.to_string())

ct = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(sparse_output=False), ["Department", "Position"]),
    ],
    remainder="drop",
)
out = ct.fit_transform(mixed)
out_df = pd.DataFrame(out, columns=ct.get_feature_names_out())

print(f"\nUkuran asli    : {mixed.shape}")
print(f"Ukuran hasil   : {out_df.shape}")
print("Nama kolom hasil:", list(out_df.columns))
print()
print(out_df.head(4).to_string())

**Penjelasan hasil:** 3 kolom numerik diskalakan dan 2 kolom kategorikal di-one-hot, sehingga 5 kolom menjadi 3 + 4 + 3 = **10** kolom. `get_feature_names_out()` memberi nama kolom berprefiks sesuai nama transformer (`num__`, `cat__`).

Argumen `remainder` menentukan nasib kolom yang tidak disebut: `"drop"` (default) membuangnya, sedangkan `"passthrough"` mempertahankannya apa adanya. Tanpa `ColumnTransformer`, kita harus memecah DataFrame, mentransformasi satu per satu, lalu menyatukan kembali, yang rawan salah dan sulit direproduksi.

## 5. Pipeline: Alur Preprocessing yang Bebas Leakage

Pipeline menyatukan preprocessing dan model dalam satu objek.

- `pipeline.fit(X_train, y_train)`: tiap transformer memanggil `fit_transform()` pada data latih, lalu model akhir `fit()`.
- `pipeline.predict(X_test)`: tiap transformer hanya memanggil `transform()`, lalu model memprediksi.

Dengan begitu data uji tidak pernah dipakai untuk mempelajari parameter preprocessing.

$$\underbrace{\text{split}(X, y)}_{1} \rightarrow \underbrace{\text{pipeline.fit}(X_{train}, y_{train})}_{2} \rightarrow \underbrace{\text{pipeline.predict}(X_{test})}_{3}$$

### 5.1 Membuat dataset campuran dengan nilai hilang dan target

Berbeda dari contoh sederhana sebelumnya, di sini kita memakai target sungguhan: `Left` (karyawan keluar atau tidak), yang dibangkitkan dari fitur-fiturnya.

In [ ]:
rng = np.random.RandomState(7)
n = 400

emp = pd.DataFrame({
    "Age": rng.randint(22, 60, n).astype(float),
    "Salary": np.round(rng.normal(60000, 15000, n), 2),
    "Experience": rng.randint(0, 25, n).astype(float),
    "Department": rng.choice(["IT", "HR", "Sales", "Finance"], n),
    "Position": rng.choice(["Junior", "Senior", "Manager"], n),
})

# Target dibangkitkan dari fitur: gaji rendah + pengalaman rendah -> lebih mungkin keluar
logit = -0.00006 * (emp["Salary"] - 60000) - 0.08 * (emp["Experience"] - 12) + (emp["Department"] == "Sales") * 0.8
prob = 1 / (1 + np.exp(-logit))
emp["Left"] = (rng.random(n) < prob).astype(int)

# Tambahkan nilai hilang (~8%)
for col in ["Age", "Salary", "Experience", "Department"]:
    emp.loc[rng.random(n) < 0.08, col] = np.nan

print("Ukuran data:", emp.shape)
print("\nNilai hilang per kolom:")
print(emp.isna().sum().to_string())
print("\nProporsi target Left:")
print(emp["Left"].value_counts(normalize=True).round(3).to_string())

**Penjelasan hasil:** data memiliki sekitar 8% nilai hilang pada beberapa kolom (termasuk kolom kategorikal `Department`) dan target biner `Left`. Perhatikan proporsi targetnya: bila sangat tidak seimbang, pertimbangkan `stratify` saat split.

### 5.2 Split dulu, baru preprocessing

In [ ]:
X = emp.drop(columns="Left")
y = emp["Left"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2024, stratify=y
)
print("Data latih:", X_train.shape, "| Data uji:", X_test.shape)

**Penjelasan:** pemisahan dilakukan **sebelum** transformasi apa pun. Jika data di-scale dulu lalu baru di-split, mean dan std dihitung dengan ikut melibatkan data uji, itulah *data leakage*.

### 5.3 Membangun pipeline lengkap

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

num_cols = ["Age", "Salary", "Experience"]
cat_cols = ["Department", "Position"]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)

print(f"Akurasi data latih: {model.score(X_train, y_train):.4f}")
print(f"Akurasi data uji  : {model.score(X_test, y_test):.4f}")
print(f"Baseline (selalu tebak kelas mayoritas): {max(y_test.mean(), 1 - y_test.mean()):.4f}")

**Penjelasan hasil:** pipeline bersarang ini menangani semuanya sekaligus: kolom numerik diisi median lalu diskalakan, kolom kategorikal diisi modus lalu di-one-hot, kemudian hasilnya masuk ke regresi logistik. Satu panggilan `fit()` dan satu panggilan `score()` cukup.

Bandingkan akurasi dengan **baseline** (selalu menebak kelas mayoritas). Model hanya berguna bila lebih baik dari baseline. Karena target dibangkitkan dengan noise, jangan berharap akurasi sangat tinggi.

### 5.4 Melihat struktur pipeline

In [ ]:
from sklearn import set_config
set_config(display="diagram")

print("Langkah utama:", [name for name, _ in model.steps])
print("Isi ColumnTransformer:", [name for name, _, _ in model.named_steps["prep"].transformers])

model   # di Colab/Jupyter, baris ini menampilkan diagram interaktif

**Penjelasan:** di Colab, ekspresi `model` pada baris terakhir menampilkan diagram HTML interaktif dari seluruh langkah. Ini berguna untuk dokumentasi alur kerja.

### 5.5 Pipeline dan tuning hyperparameter

Parameter di dalam pipeline diakses dengan format `namalangkah__parameter`, berlapis bila pipeline bersarang.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "prep__num__imputer__strategy": ["mean", "median"],
    "clf__C": [0.1, 1, 10],
}
grid = GridSearchCV(model, param_grid, cv=5)
grid.fit(X_train, y_train)

print("Parameter terbaik:", grid.best_params_)
print(f"Skor CV terbaik  : {grid.best_score_:.4f}")
print(f"Skor data uji    : {grid.score(X_test, y_test):.4f}")

**Penjelasan hasil:** `prep__num__imputer__strategy` berarti: langkah `prep` → bagian `num` → langkah `imputer` → parameter `strategy`. Karena semua preprocessing berada di dalam pipeline, setiap fold cross-validation mempelajari imputer dan scaler hanya dari data latih fold itu, sehingga tuning tetap bebas leakage.

**Praktik terbaik:** semua langkah yang melakukan `fit` pada data (imputasi, scaling, encoding, seleksi fitur) harus berada di dalam pipeline. Pipeline utuh juga bisa disimpan dengan `joblib.dump(model, "model.pkl")`.

## 6. Feature Engineering

Feature engineering terdiri dari dua aktivitas yang saling melengkapi:

1. **Feature extraction/creation:** membuat fitur baru dari fitur lama (interaksi, polinomial, binning).
2. **Feature selection:** memilih subset fitur yang paling informatif dan membuang noise.

Untuk bagian ini kita pakai data regresi buatan: 8 fitur, hanya 3 yang benar-benar informatif, sehingga kita bisa mengecek apakah metode seleksi menemukan fitur yang benar.

In [ ]:
from sklearn.datasets import make_regression

Xr, yr, coef_true = make_regression(
    n_samples=300, n_features=8, n_informative=3,
    noise=10, coef=True, random_state=0
)
Xr = pd.DataFrame(Xr, columns=[f"f{i}" for i in range(8)])
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=0)

print("Koefisien asli (0 = fitur tidak informatif):")
print(pd.Series(coef_true, index=Xr.columns).round(2).to_string())

**Penjelasan:** `make_regression` dengan `coef=True` mengembalikan koefisien sebenarnya. Fitur dengan koefisien 0 murni noise. Ini jawaban yang akan kita cocokkan dengan hasil RFE dan SelectFromModel.

### 6.1 PolynomialFeatures

Banyak hubungan di dunia nyata nonlinear. `PolynomialFeatures` membuat semua kombinasi polinomial sampai derajat tertentu. Untuk derajat 2 dengan dua fitur $[x_1, x_2]$:

$$[1,\ x_1,\ x_2,\ x_1^2,\ x_1 x_2,\ x_2^2]$$

Suku interaksi $x_1 x_2$ menangkap efek gabungan dua fitur.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from math import comb

poly = PolynomialFeatures(degree=2, include_bias=True)
Xp = poly.fit_transform(Xr_train)

p = Xr_train.shape[1]
print(f"Fitur asli      : {p}")
print(f"Fitur polinomial: {Xp.shape[1]}  (rumus C(p+d, d) = C({p}+2, 2) = {comb(p+2, 2)})")
print("\nContoh 6 nama kolom pertama:", list(poly.get_feature_names_out())[:6])

# Contoh kecil agar mudah dibaca
kecil = pd.DataFrame({"x1": [2, 3], "x2": [4, 5]})
pk = PolynomialFeatures(degree=2)
print("\nContoh kecil:")
print(pd.DataFrame(pk.fit_transform(kecil), columns=pk.get_feature_names_out()).to_string(index=False))

# Pertumbuhan jumlah fitur
for pp in [10, 50, 100]:
    print(f"p={pp:>3}: derajat 2 -> {comb(pp+2,2):>6} fitur | derajat 3 -> {comb(pp+3,3):>7} fitur")

**Penjelasan hasil:** jumlah fitur hasil mengikuti rumus kombinatorik $\binom{p+d}{d}$, yaitu 8 fitur menjadi 45. Pada contoh kecil terlihat jelas kolom `1`, `x1`, `x2`, `x1^2`, `x1 x2`, `x2^2`.

**Peringatan ledakan fitur:** pertumbuhannya sangat cepat. Dengan 100 fitur dan derajat 3 muncul ratusan ribu kolom. Karena itu `PolynomialFeatures` hampir selalu dipadukan dengan regularisasi (Ridge/Lasso) atau seleksi fitur.

### 6.2 KBinsDiscretizer

Kadang rentang nilai lebih penting daripada nilai persisnya (misalnya kelompok umur). `KBinsDiscretizer` mengubah fitur kontinu menjadi bin:

| Strategi | Cara bin dibentuk |
|:--|:--|
| `uniform` | Lebar tiap bin sama |
| `quantile` | Jumlah sampel tiap bin sama |
| `kmeans` | Berdasarkan pusat cluster K-Means |

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

umur = np.array([[18], [20], [22], [25], [30], [35], [40], [60], [75], [90]])

for strategi in ["uniform", "quantile", "kmeans"]:
    kb = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy=strategi)
    bins = kb.fit_transform(umur).ravel().astype(int)
    print(f"{strategi:>8} | batas bin: {np.round(kb.bin_edges_[0], 1)} | jumlah per bin: {np.bincount(bins)}")
    print(f"{'':>8} | hasil    : {bins}")

**Penjelasan hasil:** pada strategi `uniform`, lebar bin sama sehingga data yang menumpuk di usia muda terkumpul di satu bin dan bin lain nyaris kosong. Pada `quantile`, jumlah sampel tiap bin dibuat hampir sama. `kmeans` menyesuaikan batas dengan pengelompokan alami data.

Parameter `encode` mengatur format keluaran: `"ordinal"` (bilangan bulat) atau `"onehot"` (kolom biner). Binning bermanfaat untuk model linear karena menambah sifat nonlinear (fungsi tangga). Model pohon sudah melakukan pemisahan serupa secara internal.

### 6.3 RFE (*Recursive Feature Elimination*)

RFE melatih model, membuang fitur paling tidak penting, lalu mengulang sampai tersisa jumlah fitur yang diinginkan:

1. Latih model dengan semua $p$ fitur.
2. Urutkan fitur berdasarkan kepentingan (misalnya $|w_j|$).
3. Buang fitur paling lemah.
4. Ulangi.

Cara ini menghindari evaluasi seluruh $2^p$ kombinasi subset yang tidak mungkin dihitung.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

rfe = RFE(estimator=LinearRegression(), n_features_to_select=3)
rfe.fit(Xr_train, yr_train)

ranking = pd.DataFrame({
    "Fitur": Xr.columns,
    "Ranking": rfe.ranking_,
    "Dipilih": rfe.support_,
    "Koef asli": coef_true.round(2),
}).sort_values("Ranking")
print(ranking.to_string(index=False))

**Penjelasan hasil:** fitur dengan `Ranking = 1` adalah yang dipilih. Bandingkan dengan kolom `Koef asli`: fitur yang dipilih RFE seharusnya sama dengan fitur yang koefisien aslinya tidak nol. Ini menunjukkan RFE berhasil membedakan sinyal dari noise. Pada praktiknya jumlah fitur terbaik tidak diketahui, sehingga dapat dipakai `RFECV` yang memilihnya otomatis lewat cross-validation.

### 6.4 SelectFromModel

`SelectFromModel` melatih model **sekali**, lalu memilih fitur yang kepentingannya melewati ambang (`threshold`). Untuk model linear, kepentingan adalah $|w_j|$. Lebih cepat daripada RFE.

In [ ]:
from sklearn.feature_selection import SelectFromModel

sfm = SelectFromModel(estimator=LinearRegression(), threshold="mean")
sfm.fit(Xr_train, yr_train)

imp = pd.DataFrame({
    "Fitur": Xr.columns,
    "|coef|": np.abs(sfm.estimator_.coef_),
    "Dipilih": sfm.get_support(),
}).sort_values("|coef|", ascending=False)

print(imp.round(4).to_string(index=False))
print(f"\nAmbang (mean |coef|): {np.abs(sfm.estimator_.coef_).mean():.4f}")
print("Fitur dipilih:", list(Xr.columns[sfm.get_support()]))

**Penjelasan hasil:** fitur dengan $|coef|$ di atas rata-rata dipilih. Fitur noise memiliki koefisien mendekati 0 sehingga tersaring. Perhatikan bahwa ambang `"mean"` cukup ketat: fitur `f7` yang sebenarnya informatif tetapi berkoefisien kecil ikut terbuang, sedangkan RFE dengan `n_features_to_select=3` berhasil mempertahankannya. Jadi hasil `SelectFromModel` bergantung pada ambang. Pilihan lain: `"median"` (memilih sekitar separuh fitur) atau angka ambang buatan sendiri.

| Metode | Strategi | Kecepatan |
|:--|:--|:--|
| `RFE` | Eliminasi bertahap, model dilatih berulang | Lebih lambat |
| `SelectFromModel` | Satu kali fit lalu filter ambang | Lebih cepat |

**Seleksi di dalam pipeline:** seleksi fitur juga belajar dari data (`fit`), jadi harus berada di dalam pipeline agar tidak terjadi leakage.

In [ ]:
sel_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("select", SelectFromModel(LinearRegression(), threshold="mean")),
    ("reg", LinearRegression()),
])
sel_pipe.fit(Xr_train, yr_train)
print(f"R2 uji dengan seleksi fitur di pipeline: {sel_pipe.score(Xr_test, yr_test):.4f}")

all_feat = Pipeline([("scaler", StandardScaler()), ("reg", LinearRegression())]).fit(Xr_train, yr_train)
print(f"R2 uji tanpa seleksi (8 fitur)         : {all_feat.score(Xr_test, yr_test):.4f}")

**Penjelasan hasil:** model dengan fitur terpilih memberi R² yang hampir sama dengan model memakai semua fitur, padahal jumlah fiturnya jauh lebih sedikit. Model lebih sederhana, lebih cepat, dan lebih mudah ditafsirkan tanpa kehilangan performa.

## 7. Memilih Teknik Preprocessing yang Tepat

Tidak ada scaler yang terbaik untuk semua kasus. Panduan ringkas:

| Kondisi data | Pilihan |
|:--|:--|
| Ada outlier | `RobustScaler`, `QuantileTransformer`, `PowerTransformer` |
| Distribusi normal | `StandardScaler` atau `MinMaxScaler` |
| Distribusi miring (*skewed*) | `PowerTransformer` (Yeo-Johnson / Box-Cox) |
| Distribusi tidak diketahui | `QuantileTransformer` |
| Data sparse (banyak nol) | `MaxAbsScaler` (menjaga nol) atau `Normalizer` |
| Butuh rentang [0, 1] | `MinMaxScaler` |
| Butuh panjang vektor 1 | `Normalizer` |

Contoh: mengatasi data yang miring ke kanan (seperti pendapatan).

In [ ]:
from scipy.stats import skew
from sklearn.preprocessing import PowerTransformer, QuantileTransformer

rng = np.random.RandomState(0)
pendapatan = rng.lognormal(mean=10, sigma=1, size=(500, 1))   # miring ke kanan

hasil = {
    "Asli": pendapatan,
    "Standard": StandardScaler().fit_transform(pendapatan),
    "PowerTransformer": PowerTransformer(method="yeo-johnson").fit_transform(pendapatan),
    "QuantileTransformer": QuantileTransformer(output_distribution="normal", n_quantiles=200).fit_transform(pendapatan),
}
print("Skewness (0 = simetris):")
for nama, arr in hasil.items():
    print(f"  {nama:<20}: {skew(arr.ravel()):.3f}")

**Penjelasan hasil:** `StandardScaler` hanya menggeser dan mengubah skala, jadi kemiringan (skewness) data tidak berubah. Sebaliknya `PowerTransformer` dan `QuantileTransformer` mengubah bentuk distribusi menjadi mendekati simetris/normal, yang berguna untuk model yang mengasumsikan data berdistribusi normal.

**Model berbasis pohon** (Decision Tree, Random Forest) tidak terpengaruh skala fitur sehingga tidak butuh scaling. Mereka juga relatif tahan terhadap encoding bilangan bulat.

## Ringkasan Akhir Chapter 2

| Topik | Alat | Tujuan |
|:--|:--|:--|
| **Data hilang** | `SimpleImputer`, `KNNImputer`, `IterativeImputer` | Mengisi nilai kosong dengan estimasi |
| **Scaling** | `StandardScaler`, `MinMaxScaler`, `RobustScaler`, `Normalizer` | Menyamakan skala fitur |
| **Encoding** | `OneHotEncoder`, `OrdinalEncoder`, `LabelEncoder`, `ColumnTransformer` | Mengubah kategori menjadi angka |
| **Pipeline** | `Pipeline` | Merangkai langkah dan mencegah leakage |
| **Pembuatan fitur** | `PolynomialFeatures`, `KBinsDiscretizer` | Menambah fitur informatif |
| **Seleksi fitur** | `RFE`, `SelectFromModel` | Membuang fitur tidak informatif |

**Tiga aturan utama:**

1. **Split dulu, baru transformasi.** `fit_transform()` pada data latih, `transform()` pada data uji.
2. **Gunakan pipeline.** Semua langkah yang melakukan `fit` pada data harus berada di dalamnya.
3. **Sesuaikan preprocessing dengan model dan data.** Model berbasis jarak dan gradien butuh scaling, model berbasis pohon umumnya tidak. Pilih imputer, scaler, dan encoder berdasarkan karakteristik data.